# 08 — Fechas de las reuniones para OpenKOS

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`03-jss-eda.ipynb`](03-jss-eda.ipynb) (lectura de `meetings.xml`) ·
[ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md)
**Sustenta:** [ADR 0008](../docs/decisions/0008-adr-fechas-de-reunion-para-openkos.md)

---

> **Qué se decide aquí.** Qué *fecha de evento* se entrega a OpenKOS para cada una de las
> reuniones de las 14 series del estudio. El análisis es **estructural** (fechas y horas de
> un archivo de metadatos, no contenido de las reuniones), por lo que es admisible sobre
> todas las series, de desarrollo y de evaluación
> ([ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md)).

---

# 1. El problema

OpenKOS guarda las decisiones de una serie en un mismo espacio de trabajo y, cuando una
decisión nueva retoma un tema ya decidido, **usa la fecha de la fuente para saber cuál es la
más reciente y cuál reemplaza a cuál**. A esa fecha se la llama aquí **fecha de evento**: el
momento en que ocurrió la reunión de la que viene una decisión, no el momento en que se
procesó el archivo.

El orden importa por dos razones. Primero, porque la relación entre dos decisiones tiene
dirección: «la decisión de la reunión *c* reemplaza a la de la reunión *b*» no es lo mismo
que lo contrario. Segundo, porque el estrato **E3** del experimento pregunta justamente
cómo *evolucionó* una decisión a lo largo de las reuniones; si OpenKOS no sabe qué reunión
vino primero, esa pregunta no tiene respuesta bien definida.

El problema concreto: **las cuatro reuniones de ES2015 llevan la misma fecha y ninguna hora**.
Entregadas tal cual, OpenKOS no puede distinguir cuál ocurrió antes. Las secciones siguientes
miden si ese caso es excepcional o general, y deciden qué fecha entregar.

In [1]:
import pandas as pd

from afg.extraction.openkos.event_dates import (
    meeting_date_table,
    series_date_summary,
    study_series,
)
from afg.shared.paths import AMI_DIR

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 100)

MEETINGS_XML = AMI_DIR / "corpusResources" / "meetings.xml"

# Guarda: el notebook lee los metadatos del corpus descargado.
if not MEETINGS_XML.exists():
    raise SystemExit(
        f"No se encontro {MEETINGS_XML}.\nDescarga el corpus con `uv run afg corpus download`."
    )

SERIES = study_series()
table = meeting_date_table(MEETINGS_XML)
summary = series_date_summary(table)
print(f"Series del estudio: {len(SERIES)} (desarrollo + evaluación); reuniones: {len(table)}")

Series del estudio: 14 (desarrollo + evaluación); reuniones: 56


---

# 2. La evidencia: qué trae AMI

AMI registra, en `corpusResources/meetings.xml`, dos atributos por reunión: `dateOnly` (el
día, escrito día-mes-año, con año de dos cifras en las series ES y de cuatro en IS y TS) y
`startTime` (la hora de inicio, por ejemplo `11h03`). La tabla siguiente la produce
`meeting_date_table` (`src/afg/extraction/openkos/event_dates.py`), con una fila por
reunión. La columna `start_time_state` distingue tres situaciones del atributo `startTime`:

- `present`: tiene un valor, una hora real de inicio.
- `empty`: el atributo existe pero está vacío (`startTime=""`).
- `absent`: el atributo no existe en la reunión.

Para el análisis, `empty` y `absent` equivalen (no hay hora), pero se separan porque son
dos formas distintas de faltar y conviene registrar cuál ocurre en cada serie.

In [2]:
view = table[
    [
        "series",
        "meeting",
        "partition",
        "date_only_raw",
        "date",
        "start_time_raw",
        "start_time_state",
        "start_time",
        "event_date",
        "time_source",
    ]
].copy()
view["date"] = view["date"].dt.date
view

,series,meeting,partition,date_only_raw,date,start_time_raw,start_time_state,start_time,event_date,time_source
0,ES2015,ES2015a,development,09-03-05,2005-03-09,,empty,NaN,2005-03-09T09:00,synthetic
1,ES2015,ES2015b,development,09-03-05,2005-03-09,,empty,NaN,2005-03-09T10:00,synthetic
2,ES2015,ES2015c,development,09-03-05,2005-03-09,,empty,NaN,2005-03-09T11:00,synthetic
3,ES2015,ES2015d,development,09-03-05,2005-03-09,,empty,NaN,2005-03-09T12:00,synthetic
4,IS1004,IS1004a,development,23-11-2004,2004-11-23,11h03,present,11:03,2004-11-23T11:03,ami
5,IS1004,IS1004b,development,23-11-2004,2004-11-23,12h03,present,12:03,2004-11-23T12:03,ami
6,IS1004,IS1004c,development,23-11-2004,2004-11-23,14h13,present,14:13,2004-11-23T14:13,ami
7,IS1004,IS1004d,development,23-11-2004,2004-11-23,15h34,present,15:34,2004-11-23T15:34,ami
8,TS3009,TS3009a,development,20-10-2004,2004-10-20,NaN,absent,NaN,2004-10-20T09:00,synthetic
9,TS3009,TS3009b,development,20-10-2004,2004-10-20,NaN,absent,NaN,2004-10-20T10:00,synthetic


## 2.1 Una fecha por serie, siempre

Lo primero que se mide es cuántas fechas distintas hay dentro de cada serie.

In [3]:
dates_per_series = summary.set_index("series")["n_distinct_dates"]
print("Fechas distintas por serie (14 series):", {k: int(v) for k, v in dates_per_series.items()})
print(
    "¿Las cuatro reuniones comparten una sola fecha en las 14 series?",
    bool((dates_per_series == 1).all()),
)

Fechas distintas por serie (14 series): {'ES2015': 1, 'IS1004': 1, 'TS3009': 1, 'ES2002': 1, 'ES2008': 1, 'ES2014': 1, 'ES2016': 1, 'IS1003': 1, 'IS1006': 1, 'IS1008': 1, 'IS1009': 1, 'TS3003': 1, 'TS3005': 1, 'TS3011': 1}
¿Las cuatro reuniones comparten una sola fecha en las 14 series? True


**Resultado.** En las 14 series las cuatro reuniones tienen exactamente la misma fecha. La
fecha, por sí sola, **no ordena nada**: ES2015 no es un caso excepcional.

## 2.2 La hora de inicio: existe en unas series y no en otras

In [4]:
state = (
    table.groupby("series", sort=False)["start_time_state"]
    .agg(lambda s: ", ".join(sorted(set(s))))
    .rename("estado de startTime")
)
by_state = state.reset_index().groupby("estado de startTime")["series"].agg(list)
by_state.to_frame("series")

,series
estado de startTime,
absent,"[TS3009, TS3003, TS3005, TS3011]"
empty,"[ES2015, ES2002, ES2014, ES2016]"
present,"[IS1004, ES2008, IS1003, IS1006, IS1008, IS1009]"


In [5]:
real = summary[summary["all_real_times"]]
none_ = summary[summary["n_real_times"] == 0]
partial = summary[(summary["n_real_times"] > 0) & ~summary["all_real_times"]]
print(f"Series con hora real en las cuatro reuniones: {len(real)} -> {list(real['series'])}")
print(f"Series sin ninguna hora real              : {len(none_)} -> {list(none_['series'])}")
print(f"Series con hora real solo en algunas      : {len(partial)}")

Series con hora real en las cuatro reuniones: 6 -> ['IS1004', 'ES2008', 'IS1003', 'IS1006', 'IS1008', 'IS1009']
Series sin ninguna hora real              : 8 -> ['ES2015', 'TS3009', 'ES2002', 'ES2014', 'ES2016', 'TS3003', 'TS3005', 'TS3011']
Series con hora real solo en algunas      : 0


**Resultado.** Solo 6 de las 14 series traen la hora de inicio de sus cuatro reuniones. Las
otras 8 no traen ninguna: en cuatro de ellas el atributo está vacío y en las otras cuatro
directamente no existe (tabla anterior). No hay series con horas a medias, de modo que la
regla no necesita decidir caso por caso dentro de una serie.

## 2.3 Cuando la hora existe, ¿respeta el orden a→d?

In [6]:
cols = [
    "series",
    "partition",
    "real_times_increasing",
    "gap_ab_min",
    "gap_bc_min",
    "gap_cd_min",
    "event_dates_increasing",
]
real[cols].set_index("series")

,partition,real_times_increasing,gap_ab_min,gap_bc_min,gap_cd_min,event_dates_increasing
series,,,,,,
IS1004,development,True,60.0,130.0,81.0,True
ES2008,evaluation,True,64.0,133.0,78.0,True
IS1003,evaluation,True,74.0,122.0,82.0,True
IS1006,evaluation,True,67.0,70.0,141.0,True
IS1008,evaluation,True,66.0,121.0,84.0,True
IS1009,evaluation,True,64.0,137.0,71.0,True


In [7]:
print(
    "Series con hora real y estrictamente crecientes de a a d:",
    int(real["real_times_increasing"].sum()),
    "de",
    len(real),
)
gaps = real[["gap_ab_min", "gap_bc_min", "gap_cd_min"]].stack()
print(
    f"Separación entre inicios consecutivos: mín. {gaps.min():.0f} min, "
    f"máx. {gaps.max():.0f} min, mediana {gaps.median():.0f} min"
)

Series con hora real y estrictamente crecientes de a a d: 6 de 6
Separación entre inicios consecutivos: mín. 60 min, máx. 141 min, mediana 80 min


**Resultado.** En las 6 series con hora real, los inicios crecen estrictamente de la reunión
*a* a la *d*. Esto corrobora, con datos, que el orden a→d coincide con el orden real
cuando este es observable. Las separaciones entre inicios son contexto descriptivo (incluyen
la duración de cada reunión y los descansos) y **no se usan** como medida de tiempo en
ningún análisis posterior.

## 2.4 De dónde viene el orden a→d en las 8 series sin hora

El orden de las cuatro reuniones no se infiere de las horas: lo define el **diseño del
escenario de AMI**. El equipo de cuatro personas se reúne en secuencia: arranque
(*kick-off*), diseño funcional, diseño conceptual y diseño detallado (reuniones *a*, *b*,
*c*, *d*; ver la propuesta, sección de corpus, y
[`docs/avances/02-jss-fuente-de-datos.md`](../docs/avances/02-jss-fuente-de-datos.md) §2).
El repositorio no incluye la descripción original del escenario; la secuencia se toma como
diseño declarado del corpus. Las 6 series con hora real la confirman (sección 2.3), lo que
respalda extenderla a las que no tienen hora.

---

# 3. Alternativas consideradas

| Alternativa | Qué implicaría | Por qué se descarta |
|---|---|---|
| **No entregar hora** (solo la fecha real) | Las cuatro reuniones quedan con la misma fecha de evento. | OpenKOS no puede ordenarlas, y la dirección de reemplazo queda indefinida en las 8 de 14 series sin hora. Inutiliza el estrato E3 en esas series. |
| **Reordenar por otro criterio** (identificador, duración, contenido) | Inferir el orden a partir de otra señal. | No hay otra señal independiente: el identificador *ya* codifica a→d (sería circular) y la duración o el contenido no son evidencia de cronología. Además, leer contenido de las series de evaluación contaminaría la evaluación ([ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md)). |
| **Hora sintética que solo preserva el orden** (elegida) | Fecha real y, si falta la hora, 09:00, 10:00, 11:00 y 12:00 para *a*, *b*, *c*, *d*. | Es la única opción que da un orden estricto sin inventar una fecha. Se acepta con sus límites, descritos abajo. |

**Límites de la hora sintética.** Codifica **solo el orden**. No representa cuánto duró cada
reunión ni cuánto tiempo pasó entre una y otra: la hora «10:00» de una reunión sintética no
es un dato de AMI. Además, es una desviación declarada: en 8 de las 14 series la fecha de
evento que recibe OpenKOS contiene una hora que no ocurrió. Por eso cada reunión conserva
su `time_source` (`ami` o `synthetic`) y los resultados pueden compararse entre ambos grupos.

---

# 4. La regla y las fechas resultantes

1. **Fecha real siempre**: el `dateOnly` de AMI, leído como día-mes-año.
2. **Hora real si existe**: con `startTime`, la fecha de evento es la fecha más esa hora
   (`time_source = ami`).
3. **Hora sintética si falta**: 09:00, 10:00, 11:00 y 12:00 para *a*, *b*, *c* y *d*
   (`time_source = synthetic`). Se decide por reunión.
4. **Validación**: dentro de cada serie las fechas de evento deben ser estrictamente
   crecientes de *a* a *d*; si no, el procesamiento se detiene con un error y no se
   reordena nada.

La tabla muestra la fecha de evento que recibiría OpenKOS para cada reunión de las 14
series; la columna final indica el origen de la hora de la serie.

In [8]:
wide = table.assign(letra=table["meeting"].str[-1]).pivot(
    index="series", columns="letra", values="event_date"
)
wide = wide.loc[[s for s, _ in SERIES]]
wide.columns = [f"reunión {c}" for c in wide.columns]
wide["partición"] = [dict(SERIES)[s] for s in wide.index]
wide["origen de la hora"] = (
    table.groupby("series")["time_source"].agg(lambda s: "/".join(sorted(set(s)))).loc[wide.index]
)
wide

,reunión a,reunión b,reunión c,reunión d,partición,origen de la hora
series,,,,,,
ES2015,2005-03-09T09:00,2005-03-09T10:00,2005-03-09T11:00,2005-03-09T12:00,development,synthetic
IS1004,2004-11-23T11:03,2004-11-23T12:03,2004-11-23T14:13,2004-11-23T15:34,development,ami
TS3009,2004-10-20T09:00,2004-10-20T10:00,2004-10-20T11:00,2004-10-20T12:00,development,synthetic
ES2002,2004-12-13T09:00,2004-12-13T10:00,2004-12-13T11:00,2004-12-13T12:00,evaluation,synthetic
ES2008,2005-02-04T11:02,2005-02-04T12:06,2005-02-04T14:19,2005-02-04T15:37,evaluation,ami
ES2014,2005-03-02T09:00,2005-03-02T10:00,2005-03-02T11:00,2005-03-02T12:00,evaluation,synthetic
ES2016,2005-03-16T09:00,2005-03-16T10:00,2005-03-16T11:00,2005-03-16T12:00,evaluation,synthetic
IS1003,2004-11-19T10:35,2004-11-19T11:49,2004-11-19T13:51,2004-11-19T15:13,evaluation,ami
IS1006,2004-11-30T09:29,2004-11-30T10:36,2004-11-30T11:46,2004-11-30T14:07,evaluation,ami


In [9]:
print(
    "¿Fechas de evento estrictamente crecientes de a a d en las 14 series?",
    bool(summary["event_dates_increasing"].all()),
)
print("Reuniones con hora real (ami)       :", int((table["time_source"] == "ami").sum()))
print("Reuniones con hora sintética        :", int((table["time_source"] == "synthetic").sum()))
print(
    "Series con hora real / sintética    :",
    int(summary["all_real_times"].sum()),
    "/",
    int((~summary["all_real_times"]).sum()),
)

¿Fechas de evento estrictamente crecientes de a a d en las 14 series? True
Reuniones con hora real (ami)       : 24
Reuniones con hora sintética        : 32
Series con hora real / sintética    : 6 / 8


**Resultado.** Con la regla, las 14 series quedan con fechas de evento estrictamente
crecientes de la reunión *a* a la *d*, de modo que la validación del punto 4 no detiene
ninguna. En las 6 series con hora real, las horas son las de AMI; en las otras 8, las horas
son sintéticas y solo indican el orden. En particular, ES2015 pasa de cuatro reuniones
indistinguibles a 09:00, 10:00, 11:00 y 12:00 del mismo día.

---

# 5. Conclusión

- La fecha sola no ordena las reuniones en ninguna de las 14 series: hace falta una hora.
- AMI trae la hora real en 6 series, con orden a→d estrictamente creciente; en las otras 8
  no existe.
- Se entrega la fecha real y, cuando falta la hora, una hora sintética que solo preserva el
  orden definido por el escenario. Es una desviación declarada, trazable por `time_source`
  en el manifiesto de cada corrida.
- Ningún análisis debe usar la diferencia entre fechas de evento como medida de tiempo.

La decisión queda registrada en el
[ADR 0008](../docs/decisions/0008-adr-fechas-de-reunion-para-openkos.md); su evidencia es este
notebook y las funciones `meeting_date_table` y `series_date_summary` de
`src/afg/extraction/openkos/event_dates.py`.